# 🚀 Fine-tune BARTpho cho Tóm Tắt Bài Báo Khoa Học Tiếng Việt

**Notebook này thực hiện:**
- Kết nối Supabase để lấy ~1000 bài báo khoa học tiếng Việt
- Fine-tune model **BARTpho** (`vinai/bartpho-word`) với **LoRA (PEFT)**
- Đánh giá bằng ROUGE và demo inference
- Push model lên Hugging Face Hub (tùy chọn)

> **Môi trường:** Kaggle Notebook với GPU T4  
> **Model:** `vinai/bartpho-word`  
> **Task:** Abstractive Summarization

## 📦 Cell 1: Cài đặt các thư viện cần thiết

Chạy cell này đầu tiên để cài đặt toàn bộ dependencies.

In [1]:

# # =====================================================
# # CELL 1: CÀI ĐẶT TẤT CẢ THƯ VIỆN (ĐÃ GỘP)
# # =====================================================
# # Khuyên dùng %pip thay cho !pip trên Kaggle để tránh lỗi không tìm thấy module
# %pip install "numpy<2.0"
# %pip install -U --no-cache-dir \
#   "transformers==4.45.0" \
#   "huggingface_hub>=0.25.0" \
#   peft \
#   datasets \
#   evaluate \
#   accelerate \
#   supabase \
#   rouge-score \
#   sentencepiece

# print("✅ Đã cài đặt và nâng cấp tất cả thư viện thành công!")

In [2]:
%pip install -q --no-cache-dir \
  "transformers==4.45.0" \
  "peft>=0.13.0,<0.15.0" \
  "datasets>=2.20" \
  "evaluate>=0.4" \
  "accelerate>=0.30" \
  supabase rouge-score sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 117.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 331.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 346.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 273.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 344.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 334.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 402.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 357.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 370.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -U --no-cache-dir bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## 📚 Cell 2: Import libraries và thiết lập môi trường

In [4]:
# Cell 2: Import tất cả thư viện cần thiết
import os
import json
import random
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Torch và Transformers
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)

# PEFT / LoRA
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel
)

# Datasets và Evaluation
from datasets import Dataset, DatasetDict
import evaluate

# Supabase client
from supabase import create_client, Client

# Kaggle Secrets (để lấy API keys an toàn)
from kaggle_secrets import UserSecretsClient

# Hugging Face Hub
from huggingface_hub import login, HfApi

# ─── Thiết lập seed để kết quả reproducible ───
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─── Kiểm tra GPU ───
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# ─── Các hằng số cấu hình ───
MODEL_NAME = 'vinai/bartpho-word'           # Model BARTpho dùng word-level
MAX_INPUT_LENGTH = 1024                      # Độ dài tối đa của input (content bài báo)
MAX_TARGET_LENGTH = 300                      # Độ dài tối đa của target (abstract)
TRAIN_RATIO = 0.8                            # 80% train, 20% test
OUTPUT_DIR = '/kaggle/working/bartpho-finetuned'

print('\n✅ Import và thiết lập hoàn tất!')

2026-05-17 07:50:50.284380: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779004250.500416      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779004250.563562      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779004251.093676      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779004251.093701      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779004251.093703      57 computation_placer.cc:177] computation placer alr

🖥️  Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB

✅ Import và thiết lập hoàn tất!


## 🔌 Cell 3: Kết nối Supabase và tải dữ liệu

### Hướng dẫn thêm Kaggle Secret:
1. Vào **Kaggle Notebook** → Menu bên trái → **Add-ons** → **Secrets**
2. Thêm 2 secrets:
   - `SUPABASE_URL`: URL của project Supabase (vd: `https://xxx.supabase.co`)
   - `SUPABASE_KEY`: **anon key** hoặc **service_role key** từ Settings → API
3. Bật toggle **"Attach to notebook"** cho cả hai secrets

In [5]:
# Cell 3: Kết nối Supabase và tải dữ liệu từ bảng `papers`

# ─── Lấy credentials từ Kaggle Secrets ───
user_secrets = UserSecretsClient()

try:
    SUPABASE_URL = user_secrets.get_secret('SUPABASE_URL')
    SUPABASE_KEY = user_secrets.get_secret('SUPABASE_KEY')
    print('✅ Đã lấy secrets thành công từ Kaggle!')
except Exception as e:
    print(f'❌ Lỗi khi lấy secrets: {e}')
    print('   Hãy kiểm tra lại Kaggle Secrets theo hướng dẫn ở trên!')
    raise

# ─── Khởi tạo Supabase client ───
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print('✅ Kết nối Supabase thành công!')
# ─── 1. Kiểm tra tổng số records (không filter) ───
count_response = supabase.table('papers').select('id', count='exact').execute()
print(f'📊 Tổng số records trong bảng papers: {count_response.count}\n')
# ─── Query dữ liệu từ bảng papers ───
# Chỉ lấy records có cả content và abstract không null
print('\n📥 Đang tải dữ liệu từ Supabase (bảng papers)...')

all_records = []
PAGE_SIZE = 100   # Supabase giới hạn 1000 records/request, dùng pagination
offset = 0

while True:
    try:
        response = (
            supabase.table('papers')
            .select('title, content, abstract')
            .not_.is_('content', 'null')
            .not_.is_('abstract', 'null')
            .neq('content', '')
            .neq('abstract', '')
            .range(offset, offset + PAGE_SIZE - 1)
            .execute()
        )
        
        batch = response.data
        if not batch:
            break  # Đã lấy hết dữ liệu
            
        all_records.extend(batch)
        offset += PAGE_SIZE
        print(f'   Đã tải: {len(all_records)} records...', end='\r')
        
        # Giới hạn 1000 records như yêu cầu
        if len(all_records) >= 1000:
            all_records = all_records[:1000]
            break

    except Exception as e:
        # Xử lý lỗi RLS (Row Level Security) của Supabase
        print(f'\n⚠️  Lỗi query (có thể do RLS): {e}')
        print('   Thử dùng service_role key thay vì anon key để bypass RLS')
        raise

print(f'\n✅ Tải xong! Tổng số records hợp lệ: {len(all_records)}')

# ─── Chuyển sang DataFrame để kiểm tra ───
df = pd.DataFrame(all_records)
print(f'\n📊 Thống kê dataset:')
print(f'   Số bài báo: {len(df)}')
print(f'   Độ dài content trung bình: {df["content"].str.len().mean():.0f} ký tự')
print(f'   Độ dài abstract trung bình: {df["abstract"].str.len().mean():.0f} ký tự')
print(f'\n📄 Mẫu dữ liệu đầu tiên:')
print(f'   Title: {df["title"].iloc[0][:100] if "title" in df.columns else "N/A"}')
print(f'   Content (200 ký tự đầu): {df["content"].iloc[0][:200]}...')
print(f'   Abstract (200 ký tự đầu): {df["abstract"].iloc[0][:200]}...')

✅ Đã lấy secrets thành công từ Kaggle!
✅ Kết nối Supabase thành công!
📊 Tổng số records trong bảng papers: 1000


📥 Đang tải dữ liệu từ Supabase (bảng papers)...
   Đã tải: 994 records...
✅ Tải xong! Tổng số records hợp lệ: 994

📊 Thống kê dataset:
   Số bài báo: 994
   Độ dài content trung bình: 17406 ký tự
   Độ dài abstract trung bình: 1063 ký tự

📄 Mẫu dữ liệu đầu tiên:
   Title: Ảnh hưởng của chất trợ tương hợp glycidyl methacrylate và khoáng sét montmorillonite đến tính chất v
   Content (200 ký tự đầu): Đặt vấn đề Polybutylen adipat terephthalate (PBAT) là một polyester mềm dẻo, có độ dãn dài cao và khả năng phân hủy sinh học trong môi trường ủ phân composite [1]. Vì vậy, PBAT thường được ứng dụng tr...
   Abstract (200 ký tự đầu): Polymer blend trên cơ sở polybutylene adipate terephthalate (PBAT) kết hợp với polylactic acid (PLA) với tính chất cơ lý được cải thiện đã được chế tạo thành công trên máy đùn trục vít, sử dụng chất t...


## 🗂️ Cell 4: Chuẩn bị Dataset cho BARTpho

- Chuyển DataFrame → Hugging Face Dataset
- Train/Test split (80/20)
- Tokenize với tokenizer của `vinai/bartpho-word`

In [7]:
# Cell 4: Chuẩn bị Dataset và Tokenization
SUMMARIZATION_PROMPT = (
    "Dựa vào nội dung bài báo khoa học dưới đây, viết một đoạn tóm tắt bằng tiếng Việt. "
    "Đoạn tóm tắt phải có: câu mở đầu nêu rõ mục tiêu nghiên cứu, "
    "các câu trình bày phương pháp và kết quả chính, và câu kết luận nêu ý nghĩa. "
    "CHỈ sử dụng thông tin có trong bài. "
    "Tuyệt đối KHÔNG bịa thêm số liệu, công dụng, thời gian, địa điểm hoặc phương pháp nào không được đề cập::\n\n"
)
# ─── Load Tokenizer của BARTpho ───
print(f'📥 Đang load tokenizer từ {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print(f'✅ Tokenizer loaded! Vocab size: {tokenizer.vocab_size}')

# ─── Hàm tiền xử lý dữ liệu ───
# ─── Hàm tiền xử lý dữ liệu (ĐÃ SỬA CHO BARTPHO) ───
def preprocess_function(examples):
    """
    Tokenize cặp (content, abstract) cho BARTpho.
    Không dùng as_target_tokenizer() vì PhobertTokenizer không hỗ trợ.
    """
    prompted_contents = [SUMMARIZATION_PROMPT + content for content in examples['content']]
    # Tokenize input (content bài báo)
    model_inputs = tokenizer(
        prompted_contents,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
        return_token_type_ids=False
    )
    
    # Tokenize target (abstract) - Không cần with as_target_tokenizer()
    labels = tokenizer(
        examples['abstract'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
        return_token_type_ids=False
    )
    
    # Gán labels
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# ─── Chuyển DataFrame sang HF Dataset ───
# Chỉ giữ các cột cần thiết
df_clean = df[['content', 'abstract']].dropna().reset_index(drop=True)
print(f'\n📊 Dataset sau khi làm sạch: {len(df_clean)} mẫu')

# Tạo HF Dataset
hf_dataset = Dataset.from_pandas(df_clean)

# ─── Train/Test Split (80/20) ───
split_dataset = hf_dataset.train_test_split(
    test_size=1 - TRAIN_RATIO,
    seed=SEED
)
print(f'   Train: {len(split_dataset["train"])} mẫu')
print(f'   Test:  {len(split_dataset["test"])} mẫu')

# ─── Tokenize toàn bộ dataset ───
print('\n🔄 Đang tokenize dataset (có thể mất vài phút)...')
tokenized_dataset = split_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,
    remove_columns=['content', 'abstract'],  # Xóa cột gốc sau khi tokenize
    desc='Tokenizing'
)

print(f'\n✅ Tokenization hoàn tất!')
print(f'   Train features: {tokenized_dataset["train"].features}')

# ─── Kiểm tra độ dài token ───
train_lengths = [len(x) for x in tokenized_dataset['train']['input_ids']]
print(f'\n📏 Thống kê độ dài token (train):')
print(f'   Min: {min(train_lengths)} | Max: {max(train_lengths)} | Mean: {np.mean(train_lengths):.0f}')

📥 Đang load tokenizer từ vinai/bartpho-word...


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded! Vocab size: 64000

📊 Dataset sau khi làm sạch: 994 mẫu
   Train: 795 mẫu
   Test:  199 mẫu

🔄 Đang tokenize dataset (có thể mất vài phút)...


Tokenizing:   0%|          | 0/795 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/199 [00:00<?, ? examples/s]


✅ Tokenization hoàn tất!
   Train features: {'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}

📏 Thống kê độ dài token (train):
   Min: 104 | Max: 1024 | Mean: 1013


## 🤖 Cell 5: Load Model BARTpho + Cấu hình LoRA (PEFT)

LoRA giúp giảm đáng kể số tham số cần train, phù hợp với GPU T4 của Kaggle.

In [8]:
# Cell 5: Load Model và áp dụng LoRA
print(f'📥 Đang load model {MODEL_NAME}...')

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)

# === CẤU HÌNH GENERATION QUAN TRỌNG CHO BARTPHO ===
base_model.generation_config.decoder_start_token_id = tokenizer.bos_token_id
base_model.generation_config.forced_bos_token_id = tokenizer.bos_token_id
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
base_model.generation_config.eos_token_id = tokenizer.eos_token_id
base_model.generation_config.bos_token_id = tokenizer.bos_token_id

print(f'   ✅ Đã set generation_config đầy đủ cho BARTpho')

total_params = sum(p.numel() for p in base_model.parameters())
print(f'✅ Model loaded! Tổng tham số: {total_params / 1e6:.1f}M')

# LoRA (tăng rank lên)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,                    # ← Tăng từ 8 lên 16
    lora_alpha=32,           # ← Tăng tương ứng
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],  # ← Thêm k_proj, o_proj
    lora_dropout=0.1,
    bias='none',
    inference_mode=False
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()   # ← Thêm dòng này để kiểm tra

📥 Đang load model vinai/bartpho-word...


pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

   ✅ Đã set generation_config đầy đủ cho BARTpho
✅ Model loaded! Tổng tham số: 420.4M
trainable params: 3,538,944 || all params: 423,900,160 || trainable%: 0.8349


## ⚙️ Cell 6: Cấu hình Training Arguments

Các tham số được tối ưu đặc biệt cho GPU T4 (16GB VRAM) trên Kaggle.

In [11]:
# Cell 6: Cấu hình Seq2SeqTrainingArguments
# Tối ưu cho Kaggle T4 GPU (16GB VRAM)
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    
    learning_rate=2e-4,    
    warmup_ratio=0.1,            # ← Tăng
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    
    fp16=True,
    optim='adamw_torch_fused',
    
    # === THÊM QUAN TRỌNG ===
    label_smoothing_factor=0.1,   # ← Giúp loss không về 0 quá nhanh
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,
    
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rouge2',
    greater_is_better=True,
    save_total_limit=3,
    
    logging_steps=20,
    report_to='none',
    remove_unused_columns=False,
)
# training_args = Seq2SeqTrainingArguments(
#     output_dir=OUTPUT_DIR,
    
#     # ─── Số lần training ───
#     num_train_epochs=5,                 # 3-5 epochs là đủ cho fine-tuning
    
#     # ─── Batch size (quan trọng cho T4) ───
#     per_device_train_batch_size=4,      # 4 mẫu/batch trên mỗi GPU
#     per_device_eval_batch_size=4,       # Eval batch có thể lớn hơn
#     gradient_accumulation_steps=4,     # Tích lũy gradient để effective batch = 16
    
#     # ─── Learning rate và scheduling ───
#     learning_rate=3e-4*2,                 # LoRA thường dùng lr cao hơn full fine-tune
#     warmup_ratio=0.05,                  # 5% đầu để warmup
#     lr_scheduler_type='cosine',         # Cosine decay
#     weight_decay=0.01,                  # Regularization
    
#     # ─── Tối ưu bộ nhớ cho T4 ───
#     fp16=True,                          # Float16 giảm 50% VRAM
#     optim='adamw_torch_fused',          # AdamW tối ưu hóa
#     gradient_checkpointing=False,       # Tắt để tránh conflict với PEFT
    
#     # ─── Evaluation và lưu model ───
#     eval_strategy='epoch',        # Evaluate sau mỗi epoch
#     save_strategy='epoch',              # Lưu checkpoint sau mỗi epoch
#     load_best_model_at_end=True,        # Load model tốt nhất khi kết thúc
#     metric_for_best_model='rouge2',     # Dùng ROUGE-2 để chọn model tốt nhất
#     greater_is_better=True,
#     save_total_limit=2,                 # Chỉ giữ 2 checkpoint gần nhất
    
#     # ─── Generation ───
#     predict_with_generate=True,         # Sinh văn bản thật khi evaluate
#     generation_max_length=MAX_TARGET_LENGTH,
#     generation_num_beams=4,             # Beam search với 4 beams
    
#     # ─── Logging ───
#     logging_dir=f'{OUTPUT_DIR}/logs',
#     logging_steps=50,
#     report_to='none',                   # Không gửi lên wandb/tensorboard
    
#     # ─── Misc ───
#     seed=SEED,
#     dataloader_num_workers=2,
#     dataloader_pin_memory=True if device == 'cuda' else False,
#     remove_unused_columns=False,        # Để tránh lỗi với PEFT
# )

print('✅ Cấu hình training:')
print(f'   Epochs:               {training_args.num_train_epochs}')
print(f'   Batch size (train):   {training_args.per_device_train_batch_size}')
print(f'   Gradient accum:       {training_args.gradient_accumulation_steps}')
print(f'   Effective batch:      {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'   Learning rate:        {training_args.learning_rate}')
print(f'   FP16:                 {training_args.fp16}')

✅ Cấu hình training:
   Epochs:               10
   Batch size (train):   4
   Gradient accum:       4
   Effective batch:      16
   Learning rate:        0.0002
   FP16:                 True


## 🏋️ Cell 7: Khởi tạo Trainer và Bắt đầu Training

In [12]:
# Cell 7: Khởi tạo Seq2SeqTrainer và bắt đầu training

# ─── Load ROUGE metric ───
rouge_metric = evaluate.load('rouge')

# ─── Hàm tính metrics trong quá trình evaluate ───
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Loại bỏ chuỗi rỗng
    decoded_preds = [pred.strip() if pred.strip() else "Tóm tắt" for pred in decoded_preds]
    
    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False
    )
    return {
        'rouge1': round(result['rouge1'] * 100, 4),
        'rouge2': round(result['rouge2'] * 100, 4),
        'rougeL': round(result['rougeL'] * 100, 4),
    }
# ─── Tạo DataCollator (dynamic padding) ───
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,  # Padding labels với -100 (bị bỏ qua khi tính loss)
    pad_to_multiple_of=8      # Tối ưu tensor operations
)

# ─── Khởi tạo Seq2SeqTrainer ───
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # Dừng sớm nếu không cải thiện
)

# ─── Bắt đầu Training! ───
print('🚀 Bắt đầu training BARTpho với LoRA...')
print(f'   Tổng steps ước tính: {len(tokenized_dataset["train"]) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}')
print('=' * 60)

train_result = trainer.train()

print('\n' + '=' * 60)
print('✅ Training hoàn tất!')
print(f'   Train Loss cuối: {train_result.training_loss:.4f}')
print(f'   Tổng thời gian:  {train_result.metrics["train_runtime"] / 60:.1f} phút')

# Lưu kết quả training
trainer.save_model(OUTPUT_DIR)
trainer.log_metrics('train', train_result.metrics)
trainer.save_metrics('train', train_result.metrics)
print(f'\n💾 Model đã được lưu tại: {OUTPUT_DIR}')

🚀 Bắt đầu training BARTpho với LoRA...
   Tổng steps ước tính: 490


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,7.032600,5.647242,38.467400,11.612600,24.221400
2,6.095600,5.213413,45.293900,15.106500,28.440000
3,5.706600,5.019649,50.471700,18.690600,29.078300
4,5.337900,4.912945,51.612100,19.734900,28.718700
5,5.236300,4.853102,55.594000,23.797400,29.983500
6,5.153400,4.819512,57.082100,25.008100,30.360400
7,5.161900,4.798395,59.252800,26.439600,30.816700
8,5.095100,4.785531,61.391500,27.952500,31.572200
9,5.069300,4.784823,61.086300,27.709900,31.497900
10,5.094700,4.784278,60.596700,27.477000,31.476800



✅ Training hoàn tất!
   Train Loss cuối: 5.4417
   Tổng thời gian:  51.8 phút
***** train metrics *****
  epoch                    =       10.0
  total_flos               = 16206447GF
  train_loss               =     5.4417
  train_runtime            = 0:51:47.69
  train_samples_per_second =      2.558
  train_steps_per_second   =       0.08

💾 Model đã được lưu tại: /kaggle/working/bartpho-finetuned


## 📊 Cell 8: Đánh giá Model bằng ROUGE

In [14]:
# Cell 8: Đánh giá Model trên Test Set

print('📊 Đang đánh giá model trên test set...')
print('   (Quá trình này sinh văn bản cho toàn bộ test set, có thể mất vài phút)')

# Chạy evaluation
eval_results = trainer.evaluate(
    eval_dataset=tokenized_dataset['test'],
    metric_key_prefix='eval'
)

print('\n' + '═' * 50)
print('📈 KẾT QUẢ ĐÁNH GIÁ (ROUGE Scores)')
print('═' * 50)
print(f"   ROUGE-1 : {eval_results.get('eval_rouge1', 0):.2f}%")
print(f"   ROUGE-2 : {eval_results.get('eval_rouge2', 0):.2f}%")
print(f"   ROUGE-L : {eval_results.get('eval_rougeL', 0):.2f}%")
print(f"   Eval Loss: {eval_results.get('eval_loss', 0):.4f}")
print('═' * 50)

# ─── Giải thích kết quả ───
print('\n💡 Giải thích ROUGE Score:')
print('   ROUGE-1: Tỷ lệ unigrams (từ đơn) khớp với reference')
print('   ROUGE-2: Tỷ lệ bigrams (cặp từ) khớp với reference')
print('   ROUGE-L: Dựa trên Longest Common Subsequence')
print('   → Thông thường ROUGE-2 > 10% là chấp nhận được cho tiếng Việt')

# Lưu kết quả evaluation
trainer.save_metrics('eval', eval_results)
print(f'\n💾 Kết quả đã lưu tại {OUTPUT_DIR}/eval_results.json')

📊 Đang đánh giá model trên test set...
   (Quá trình này sinh văn bản cho toàn bộ test set, có thể mất vài phút)

══════════════════════════════════════════════════
📈 KẾT QUẢ ĐÁNH GIÁ (ROUGE Scores)
══════════════════════════════════════════════════
   ROUGE-1 : 61.39%
   ROUGE-2 : 27.95%
   ROUGE-L : 31.57%
   Eval Loss: 4.7855
══════════════════════════════════════════════════

💡 Giải thích ROUGE Score:
   ROUGE-1: Tỷ lệ unigrams (từ đơn) khớp với reference
   ROUGE-2: Tỷ lệ bigrams (cặp từ) khớp với reference
   ROUGE-L: Dựa trên Longest Common Subsequence
   → Thông thường ROUGE-2 > 10% là chấp nhận được cho tiếng Việt

💾 Kết quả đã lưu tại /kaggle/working/bartpho-finetuned/eval_results.json


## 🧪 Cell 9: Test Inference - Demo với ví dụ thực tế

In [15]:
# Cell 9 - generate_summary ĐÃ SỬA (áp dụng cho cả BART5 và ViT5)
def generate_summary(text: str, max_new_tokens: int = 256, num_beams: int = 4) -> str:
    prompted_text = SUMMARIZATION_PROMPT + text

    inputs = tokenizer(
        prompted_text,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
        return_tensors='pt',
        return_token_type_ids=False
    ).to(device)

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens, 
            min_new_tokens=50,             
            num_beams=num_beams,
            early_stopping=False,         
            no_repeat_ngram_size=3,
            length_penalty=2.0,            

            # Chỉ thêm cho BART5, bỏ cho ViT5:
            forced_bos_token_id=tokenizer.bos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Loại bỏ input tokens khỏi output (encoder-decoder thường không cần,
    # nhưng thêm để an toàn)
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return summary.strip()

# ─── Chọn ngẫu nhiên 3 ví dụ từ test set ───
print('🧪 DEMO INFERENCE - Tóm tắt 3 bài báo ngẫu nhiên từ test set')
print('=' * 70)

test_indices = random.sample(range(len(split_dataset['test'])), min(3, len(split_dataset['test'])))

for i, idx in enumerate(test_indices, 1):
    example = split_dataset['test'][idx]
    content = example['content']
    ground_truth = example['abstract']
    
    # Sinh tóm tắt
    generated = generate_summary(content)
    
    # Tính ROUGE cho ví dụ này
    single_rouge = rouge_metric.compute(
        predictions=[generated],
        references=[ground_truth]
    )
    
    print(f'\n📄 Ví dụ #{i} (Test index: {idx})')
    print(f'   Input (200 ký tự đầu): {content[:1000]}...')
    print(f'   ─────────────────────────────────────')
    print(f'   🤖 Tóm tắt sinh ra:   {generated}')
    print(f'   ✅ Ground truth:       {ground_truth[:300]}')
    print(f'   📊 ROUGE-2 ví dụ này: {single_rouge["rouge2"]*100:.2f}%')
    print()

print('=' * 70)
print('✅ Demo inference hoàn tất!')

🧪 DEMO INFERENCE - Tóm tắt 3 bài báo ngẫu nhiên từ test set

📄 Ví dụ #1 (Test index: 163)
   Input (200 ký tự đầu): Đặt vấn đề Ngày nay, vật liệu nano được ứng dụng rộng rãi trong nhiều lĩnh vực đặc biệt như y sinh, xử lý các chất ô nhiễm và điện tử, trong đó, HA được biết như vật liệu có các ứng dụng đa dạng trong hấp phụ các chất ô nhiễm trong môi trường nước và y sinh. HA được tổng hợp từ các nguồn nguyên liệu và phương pháp tổng hợp khác nhau sẽ ảnh hưởng đến độ tinh khiết, diện tích bề mặt riêng và hình dạng của chúng. HA được tổng hợp từ vỏ sò biển có dạng hình que và kích thước tinh thể ~101 nm [1]. Để kiểm soát hình thái và kích thước hạt HA, dịch chiết trái cây có thể được bổ sung trong quá trình tổng hợp [2]. Với nguồn tiền chất từ hỗn hợp bột xương cá da trơn và xương động vật đã tạo ra HA có độ kết tinh và kích thước tinh thể trung bình là 80,42% và 27,3 nm [3]. Vỏ trứng cũng được sử dụng để tổng hợp HA bằng phương pháp thủy nhiệt kết hợp vi sóng ở công suất 800 W, vật liệu

In [16]:
## =====================================================
# CELL 8.5: ĐÁNH GIÁ BERTSCORE (NGỮ NGHĨA)
# =====================================================
print("📊 Đang tính BERTScore trên toàn bộ test set...")
print("   (Sử dụng PhoBERT-base - phù hợp nhất với tiếng Việt)")

# Sinh tóm tắt cho tất cả mẫu test
predictions = []
references = []

for i, example in enumerate(split_dataset['test']):
    content = example['content']
    ground_truth = example['abstract']
    
    generated = generate_summary(content)   # hàm đã có ở Cell 9
    
    predictions.append(generated)
    references.append(ground_truth)
    
    if (i + 1) % 50 == 0:
        print(f"   Đã sinh {i+1}/{len(split_dataset['test'])} tóm tắt...")

# Tính BERTScore
bertscore_metric = evaluate.load("bertscore")

results = bertscore_metric.compute(
    predictions=predictions,
    references=references,
    model_type="microsoft/mdeberta-v3-base", 
    lang="vi",
    batch_size=16,
    rescale_with_baseline=False,        
    num_layers=12
)

# Kết quả trung bình
print("\n" + "="*60)
print("🎯 KẾT QUẢ BERTSCORE (PhoBERT-base)")
print("="*60)
print(f"   Precision : {np.mean(results['precision']):.4f}")
print(f"   Recall    : {np.mean(results['recall']):.4f}")
print(f"   F1        : {np.mean(results['f1']):.4f}")
print("="*60)

# Lưu kết quả
import json
with open(f"{OUTPUT_DIR}/bertscore_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "bertscore_precision": float(np.mean(results['precision'])),
        "bertscore_recall": float(np.mean(results['recall'])),
        "bertscore_f1": float(np.mean(results['f1'])),
        "model_type": "vinai/phobert-base"
    }, f, ensure_ascii=False, indent=2)

print(f"💾 Đã lưu kết quả BERTScore tại: {OUTPUT_DIR}/bertscore_results.json")

📊 Đang tính BERTScore trên toàn bộ test set...
   (Sử dụng PhoBERT-base - phù hợp nhất với tiếng Việt)
   Đã sinh 50/199 tóm tắt...
   Đã sinh 100/199 tóm tắt...
   Đã sinh 150/199 tóm tắt...


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]


🎯 KẾT QUẢ BERTSCORE (PhoBERT-base)
   Precision : 0.8691
   Recall    : 0.8802
   F1        : 0.8745
💾 Đã lưu kết quả BERTScore tại: /kaggle/working/bartpho-finetuned/bertscore_results.json


## 💾 Cell 10: Lưu Model Cục bộ và Hướng dẫn Sử dụng

**Cách tải model và sử dụng sau này:**

In [17]:
# Cell 11: Lưu model ra file và hướng dẫn sử dụng sau này

import shutil

# ─── Lưu LoRA adapter ───
LORA_SAVE_PATH = '/kaggle/working/bartpho-lora-adapter'
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f'✅ LoRA adapter đã lưu tại: {LORA_SAVE_PATH}')

# ─── Nén thành zip để tải về ───
zip_path = '/kaggle/working/bartpho-lora-adapter.zip'
shutil.make_archive(
    '/kaggle/working/bartpho-lora-adapter',
    'zip',
    LORA_SAVE_PATH
)
print(f'📦 Đã nén thành: {zip_path}')
print(f'   → Vào Kaggle Output tab để tải file .zip về máy')

# ─── Hướng dẫn sử dụng model sau này ───
print('\n' + '=' * 60)
print('📖 HƯỚNG DẪN SỬ DỤNG MODEL SAU KHI TRAINING')
print('=' * 60)

usage_code = '''
# Cách 1: Load từ thư mục local (sau khi tải zip về)
# ─────────────────────────────────────────────────
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "vinai/bartpho-word"
ADAPTER_PATH = "./bartpho-lora-adapter"  # Giải nén zip vào đây

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# Sinh tóm tắt
def summarize(text, max_length=256):
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    with torch.no_grad():
        output = model.generate(**inputs, max_length=max_length, num_beams=4)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Ví dụ sử dụng
text = "Nội dung bài báo khoa học cần tóm tắt..."
summary = summarize(text)
print(f"Tóm tắt: {summary}")

# Cách 2: Load từ Hugging Face Hub (sau khi push)
# ─────────────────────────────────────────────────
# model = PeftModel.from_pretrained(base_model, "your-username/bartpho-word-finetuned")
'''

print(usage_code)
trainable_params = 3_538_944
# ─── Tổng kết ───
print('\n' + '🎉 ' * 20)
print('\n✨ NOTEBOOK HOÀN TẤT! Tổng kết:')
print(f'   ✅ Đã load {len(df_clean)} bài báo từ Supabase')
print(f'   ✅ Train: {len(split_dataset["train"])} | Test: {len(split_dataset["test"])} mẫu')
print(f'   ✅ Fine-tuned BARTpho với LoRA ({trainable_params/1e6:.2f}M params)')
print(f'   ✅ Đánh giá ROUGE trên test set')
print(f'   ✅ Model lưu tại: {OUTPUT_DIR}')
print(f'\n📥 Tải model về: /kaggle/working/bartpho-lora-adapter.zip')

✅ LoRA adapter đã lưu tại: /kaggle/working/bartpho-lora-adapter
📦 Đã nén thành: /kaggle/working/bartpho-lora-adapter.zip
   → Vào Kaggle Output tab để tải file .zip về máy

📖 HƯỚNG DẪN SỬ DỤNG MODEL SAU KHI TRAINING

# Cách 1: Load từ thư mục local (sau khi tải zip về)
# ─────────────────────────────────────────────────
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "vinai/bartpho-word"
ADAPTER_PATH = "./bartpho-lora-adapter"  # Giải nén zip vào đây

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

# Sinh tóm tắt
def summarize(text, max_length=256):
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    with torch.no_grad():
        output = model.generate(**inputs, max_length=max_length, num_beams=4)
    return tokenizer.decode(output[0], skip